In [1]:
!git clone https://github.com/nguyenlethienlyy/fine-tuning-intent-detection-model.git
%cd fine-tuning-intent-detection-model
!pip install transformers datasets accelerate

Cloning into 'fine-tuning-intent-detection-model'...
remote: Enumerating objects: 59, done.
remote: Counting objects: 100% (59/59), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 59 (delta 19), reused 48 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (59/59), 204.20 KiB | 1.23 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/fine-tuning-intent-detection-model


In [ ]:
import yaml
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)
from sklearn.metrics import accuracy_score


# ================= LOAD CONFIG =================
with open("configs/train.yaml", "r") as f:
    config = yaml.safe_load(f)

MODEL_NAME = config["model"]["name"]
NUM_LABELS = config["model"]["num_labels"]

TRAIN_PATH = config["data"]["train_path"]
TEST_PATH = config["data"]["test_path"]
MAX_LENGTH = config["data"]["max_length"]

TRAINING_CONFIG = config["training"]


# ================= METRICS =================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"accuracy": accuracy_score(labels, preds)}


# ================= LOAD DATA =================
print("Loading data...")
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train size:", len(train_df))
print("Test size:", len(test_df))


# ================= DATASET =================
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)


# ================= TOKENIZER =================
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )

print("Tokenizing...")
train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset = train_dataset.rename_column("label", "labels")
test_dataset = test_dataset.rename_column("label", "labels")

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])


# ================= MODEL =================
print("Loading model...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS
)


# ================= TRAINING ARGS =================
training_args = TrainingArguments(
    output_dir=TRAINING_CONFIG["output_dir"],
    per_device_train_batch_size=TRAINING_CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=TRAINING_CONFIG["per_device_eval_batch_size"],
    num_train_epochs=TRAINING_CONFIG["num_train_epochs"],
    learning_rate=TRAINING_CONFIG["learning_rate"],
    weight_decay=TRAINING_CONFIG["weight_decay"],
    evaluation_strategy=TRAINING_CONFIG["evaluation_strategy"],
    save_strategy=TRAINING_CONFIG["save_strategy"],
    logging_dir=TRAINING_CONFIG["logging_dir"],
    logging_steps=TRAINING_CONFIG["logging_steps"],
    load_best_model_at_end=TRAINING_CONFIG["load_best_model_at_end"],
    metric_for_best_model=TRAINING_CONFIG["metric_for_best_model"],
    report_to=TRAINING_CONFIG["report_to"],
)


# ================= TRAINER =================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

Loading data...
Train size: 10003
Test size: 3080
Loading tokenizer...
Tokenizing...


Map:   0%|          | 0/10003 [00:00<?, ? examples/s]

Map:   0%|          | 0/3080 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Loading model...


In [6]:
# Train
print("Training...")
trainer.train()

Training...


Epoch,Training Loss,Validation Loss,Accuracy
1,2.223100,2.021917,0.707792
2,1.185000,1.086990,0.844156
3,0.826200,0.881485,0.871429


TrainOutput(global_step=1878, training_loss=1.8776879356311151, metrics={'train_runtime': 427.7164, 'train_samples_per_second': 70.161, 'train_steps_per_second': 4.391, 'total_flos': 987627072862080.0, 'train_loss': 1.8776879356311151, 'epoch': 3.0})

In [7]:
# Save model
print("Saving model...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("Done.")

Saving model...
Done.


In [8]:
# Evaluate
print("Evaluating...")
results = trainer.evaluate()
print(results)


Evaluating...


{'eval_loss': 0.8814848065376282, 'eval_accuracy': 0.8714285714285714, 'eval_runtime': 12.0482, 'eval_samples_per_second': 255.64, 'eval_steps_per_second': 16.019, 'epoch': 3.0}


In [9]:
import shutil
shutil.make_archive("checkpoint77-1000-40", 'zip', "checkpoint")

'/content/fine-tuning-intent-detection-model/checkpoint77-1000-40.zip'